# ***02 · Simulación Monte Carlo: tres casos***

**Objetivo:** puntos 4 y 5 del plan. Implementación del algoritmo de aceptación-rechazo y análisis de la distribución simétrica (solo fotón), de una asimetría propuesta y de las corrientes neutras en $\sqrt{s}=m_Z$.

## ***Algoritmo de aceptación-rechazo***

1. proponer un valor aleatorio $x\in[-1,1]$
2. proponer una altura aleatoria $y\in[0,f_{\max}]$
3. si $y\le f(x)$, aceptar el evento
4. si $y>f(x)$, rechazar y repetir

La eficiencia es $\eta=\dfrac{\text{área bajo }f(x)}{\text{área del rectángulo}}$. La implementación está en `src/fbasym/sampling.py`; propone 20 000 puntos por iteración, de modo que escala a millones de eventos.

In [ ]:
import sys
sys.path.append("../src")          # permite importar el paquete fbasym sin instalarlo

import numpy as np
import matplotlib.pyplot as plt
from fbasym import *
from fbasym import plots

n_events = 10000
np.random.seed(42)

def resumen(eventos, eficiencia, A, B):
    afb = asimetria_fb(eventos)
    chi2, ndof, _, _ = chi2_y_pulls(eventos, A, B)
    print(f"eventos generados: {len(eventos)}")
    print(f"eficiencia: {eficiencia:.2f}%")
    print(f"media de x: {np.mean(eventos):.5f}")
    print(f"desviación estándar: {np.std(eventos):.5f}")
    print(f"A_FB simulada = {afb:+.4f} +- {error_afb(afb, len(eventos)):.4f}   |   teórica = {3*B/(8*A):+.4f}")
    print(f"chi2/ndof = {chi2:.1f}/{ndof} = {chi2/ndof:.3f}   (cerca de 1 --> buen ajuste)")

## ***Caso 1: distribución simétrica (solo fotón)***

**modelo:** $f(x)=1+x^2$, es decir $A=1$ y $B=0$. Se espera media de $x$ cercana a 0, forma de "U" y $A_{FB}\approx0$. En la gráfica de residuos, casi todos los puntos deben quedar entre las líneas verdes ($\pm2\sigma$).

In [ ]:
A_sim, B_sim = 1.0, 0.0
ev_sim, ef_sim = rejection_sampling(A_sim, B_sim, n_events)
resumen(ev_sim, ef_sim, A_sim, B_sim)

fig = plots.figura_caso(ev_sim, A_sim, B_sim, r'teoría: $f(x) = 1 + x^2$',
                        f'Distribución simétrica (solo fotón): {n_events} eventos')
fig.savefig("../figures/02_caso1_simetrico.png", dpi=150)
plt.show()

## ***Caso 2: asimetría propuesta***

Para aislar el efecto del término lineal se proponen $A_{ideal}=1$ y $B_{ideal}=0.5$, con $A_{FB}=\tfrac{3}{8}(0.5)=0.1875$. No es un cálculo físico sino una suposición para visualizar el efecto: la curva se inclina hacia $x>0$.

In [ ]:
A_ideal, B_ideal = 1.0, 0.5
ev_ideal, ef_ideal = rejection_sampling(A_ideal, B_ideal, n_events)
resumen(ev_ideal, ef_ideal, A_ideal, B_ideal)

fig = plots.figura_caso(ev_ideal, A_ideal, B_ideal,
                        r'suposición: $f(x) = A_{ideal}(1 + x^2) + B_{ideal}\,x$',
                        f'Asimetría propuesta: {n_events} eventos')
fig.savefig("../figures/02_caso2_asimetria_propuesta.png", dpi=150)
plt.show()

## ***Caso 3: corrientes neutras en el polo de la $Z$***

Con $A_{NC}\approx168$ y $B_{NC}\approx7.4$, la razón $B/A$ es pequeña: la distribución es casi simétrica y la asimetría es sutil ($A_{FB}\approx0.017$).

In [ ]:
A_NC, B_NC = coeficientes(M_Z)
ev_nc, ef_nc = rejection_sampling(A_NC, B_NC, n_events)
print(f"A_NC = {A_NC:.4f}   B_NC = {B_NC:.4f}")
resumen(ev_nc, ef_nc, A_NC, B_NC)

fig = plots.figura_caso(ev_nc, A_NC, B_NC,
                        r'teórico: $f(x) \propto A_{NC}(1 + x^2) + B_{NC}\,x$',
                        f'Corrientes neutras, $\\sqrt{{s}}=m_Z$: {n_events} eventos')
fig.savefig("../figures/02_caso3_corrientes_neutras.png", dpi=150)
plt.show()

Con 10 000 eventos el error estadístico de $A_{FB}$ es $\approx 1/\sqrt{N}=0.01$, comparable a la señal ($0.017$). Por ello la $A_{FB}$ simulada puede diferir de la teórica sin que exista ningún error en el código; este efecto se estudia en el notebook 04.